# Load Weekly Results and Append to Initial Data

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

In [ ]:
# DEFINE THE WEEK BEING UPDATED

week = "w9"

In [3]:
#Define the functions
functions = [
    "function_1", "function_2", "function_3", "function_4",
    "function_5", "function_6", "function_7", "function_8"
]

# Load the .npy file

def load_npy_files(function_name):
    initial_inputs = np.load(f'./Data/{function_name}/initial_inputs.npy')
    initial_outputs = np.load(f'./Data/{function_name}/initial_outputs.npy')
    return initial_inputs, initial_outputs

In [6]:
# Review the file contents

for function_name in functions:
    print(f"Function: {function_name}")
    X, y = load_npy_files(function_name)
    n = len(y)
    print(f"{function_name}: X{X.shape}  y{y.shape}  n = {n}")
    print("---------------------")

Function: function_1
function_1: X(18, 2)  y(18,)  n = 18
---------------------
Function: function_2
function_2: X(18, 2)  y(18,)  n = 18
---------------------
Function: function_3
function_3: X(23, 3)  y(23,)  n = 23
---------------------
Function: function_4
function_4: X(38, 4)  y(38,)  n = 38
---------------------
Function: function_5
function_5: X(28, 4)  y(28,)  n = 28
---------------------
Function: function_6
function_6: X(28, 5)  y(28,)  n = 28
---------------------
Function: function_7
function_7: X(38, 6)  y(38,)  n = 38
---------------------
Function: function_8
function_8: X(48, 8)  y(48,)  n = 48
---------------------


In [8]:
# Extract the latest itertion of data inputs for the returned files
# The files contain all the prior iterations and only need the latest appended values

#Read the input data points provided for prior estimate
with open(f'./Weekly_updates/{week}_inputs.txt', "r") as file:
    text = file.read()
print("Updated inputs file:")
#print(text)

# Extract contents of each array(...)
matches = re.findall(r'array\(\[(.*?)\]\)', text, re.DOTALL)

arrays = [np.fromstring(m, sep=',') for m in matches]
arrays = arrays[-8:]

for i, arr in enumerate(arrays):
    print(f"Array {i+1}:")
    print(arr)
    print("Shape:", arr.shape)
    


Updated inputs file:
Array 1:
[0.645355 0.665341]
Shape: (2,)
Array 2:
[0.708809 0.892037]
Shape: (2,)
Array 3:
[0.366844 0.433619 0.499863]
Shape: (3,)
Array 4:
[0.392333 0.353703 0.398126 0.437857]
Shape: (4,)
Array 5:
[1.   1.   1.   0.97]
Shape: (4,)
Array 6:
[0.517446 0.167864 0.661411 0.665938 0.143036]
Shape: (5,)
Array 7:
[0.130864 0.149666 0.27462  0.29939  0.345033 0.643752]
Shape: (6,)
Array 8:
[0.107542 0.160528 0.109353 0.096175 0.97087  0.558758 0.174878 0.344019]
Shape: (8,)


In [9]:
# Read the output data points that result from prior estimated inputs
# The outputs file also contains all prior iteration results and only the latest required

with open(f'./Weekly_updates/{week}_outputs.txt', "r") as file:
    text = file.read()
print("Outputs for:", week)
#print(text)

# Extract all numeric values regardless of format - the format of the date in returned file changed in Week 5!
# Handles: np.float64(x), plain lists [x, y, z], and bare numbers

all_values = []
for line in text.strip().split('\n'):
    line = line.strip()
    if not line:
        continue
    # Strip np.float64() wrappers then parse as a list
    cleaned = re.sub(r'np\.float64\(([^)]+)\)', r'\1', line)
    try:
        parsed = ast.literal_eval(cleaned)
        if isinstance(parsed, (list, tuple)):
            all_values.extend([float(v) for v in parsed])
        else:
            all_values.append(float(parsed))
    except Exception:
        # fallback: extract any numbers from the line
        nums = re.findall(r'[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?', cleaned)
        all_values.extend([float(n) for n in nums])

# Convert to NumPy array
Y_new = np.array(all_values, dtype=float)

Y_new = Y_new[-8:]
print(Y_new)
print(type(Y_new))
print(Y_new.shape)

Outputs for: w9
[ 1.12361190e-02  5.09792016e-01 -9.45662574e-03  3.04899849e-01
  8.11543472e+03 -4.90042077e-01  2.90977509e+00  9.96973373e+00]
<class 'numpy.ndarray'>
(8,)


In [14]:
# Iterate through the functions to add the new inputs and outputs to the data files

i = 0
for function_name in functions:
    print(f"Function: {function_name}")
    X_orig, Y_orig = load_npy_files(function_name)

    X_updated = np.vstack((X_orig, arrays[i]))
    np.save(f'./Data/{function_name}/initial_inputs_test.npy', X_updated)
    X_updated_saved = np.load(f'./Data/{function_name}/initial_inputs_test.npy')
    print("X Inputs:")
    print("Orig shape:", X_orig.shape)
    print("Updated shape:", X_updated_saved.shape)
    print (pd.DataFrame(X_orig).tail(2))
    print(pd.DataFrame(X_updated_saved).tail(3))
    
    Y_updated = np.append(Y_orig, Y_new[i])
    np.save(f'./Data/{function_name}/initial_outputs_test.npy', Y_updated)
    Y_updated_saved = np.load(f'./Data/{function_name}/initial_outputs_test.npy')
    print("Y outputs")
    print("Orig shape:", Y_orig.shape)
    print("Updated shape:", Y_updated_saved.shape)
    print (pd.DataFrame(Y_orig).tail(2))
    print(pd.DataFrame(Y_updated_saved).tail(3))
    print("---------------------")
    i=i+1


Function: function_1
X Inputs:
Orig shape: (18, 2)
Updated shape: (19, 2)
           0         1
16  0.696829  0.577612
17  0.602765  0.773888
           0         1
16  0.696829  0.577612
17  0.602765  0.773888
18  0.645355  0.665341
Y outputs
Orig shape: (18,)
Updated shape: (19,)
               0
16  7.474097e-06
17  6.618177e-16
               0
16  7.474097e-06
17  6.618177e-16
18  1.123612e-02
---------------------
Function: function_2
X Inputs:
Orig shape: (18, 2)
Updated shape: (19, 2)
           0         1
16  0.653266  0.934673
17  0.718593  0.874372
           0         1
16  0.653266  0.934673
17  0.718593  0.874372
18  0.708809  0.892037
Y outputs
Orig shape: (18,)
Updated shape: (19,)
           0
16  0.483447
17  0.601905
           0
16  0.483447
17  0.601905
18  0.509792
---------------------
Function: function_3
X Inputs:
Orig shape: (23, 3)
Updated shape: (24, 3)
           0         1         2
21  0.215348  0.376669  0.506634
22  0.341940  0.383375  0.436052
     

In [17]:
# Copy the test files to the merged versions and create CSV copies

for function_name in functions:
    print(f"Function: {function_name}")

    X = np.load(f'./Data/{function_name}/initial_inputs_test.npy')
    np.save(f'./Data/{function_name}/merged_inputs.npy', X)
    np.savetxt(f'./Data/{function_name}/input.csv', X, delimiter=',', fmt='%.10f')
    
    Y = np.load(f'./Data/{function_name}/initial_outputs_test.npy')
    np.save(f'./Data/{function_name}/merged_outputs.npy', Y)
    np.savetxt(f'./Data/{function_name}/output.csv', Y, delimiter=',')
    print("---------------------")


Function: function_1
---------------------
Function: function_2
---------------------
Function: function_3
---------------------
Function: function_4
---------------------
Function: function_5
---------------------
Function: function_6
---------------------
Function: function_7
---------------------
Function: function_8
---------------------
